# Laboratorio 5: Modelos de lenguaje

En este laboratorio se construyen modelos de lenguaje basados en n-gramas usando el texto de *Don Quijote de la Mancha*.  
El objetivo es preparar el corpus, construir modelos unigrama, bigrama y trigrama, aplicar suavizado, evaluar con perplejidad y probar una función simple de autocompletado.

## Importación de librerías

Primero se importan las librerías necesarias para leer el texto, limpiar el corpus, contar palabras y separar los datos en entrenamiento, validación y prueba.

In [3]:
# Librerías principales para manejo de texto, conteos y división de datos
import re
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

## Carga del corpus

Se carga el texto completo de *Don Quijote de la Mancha*.  
Antes de procesarlo, se muestra una pequeña parte para verificar que el archivo se leyó correctamente.

In [1]:
# Cargamos el texto completo de Don Quijote
with open("don-quijote.txt", "r", encoding="utf-8") as file:
    texto = file.read()

# Mostramos una parte pequeña del texto para verificar que se cargó bien
print(texto[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: Don Quijote

Author: Miguel de Cervantes Saavedra

Posting Date: April 27, 2010 [EBook #2000]
Release Date: December, 1999

Language: Spanish


*** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE ***




Produced by an anonymous Project Gutenberg volunteer. Text
file corrections and new HTML file by Joaquin Cuenca Abela.











El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso hidalgo de la Mancha,
compuesto por Miguel de Cervantes Saavedra, tasaron cada


## Limpieza básica

En esta parte solo se normalizan espacios y saltos de línea.  
No se eliminan stopwords ni se aplica lematización, porque el objetivo es conservar el orden natural de las palabras para construir modelos de lenguaje.

In [4]:
# Reemplazamos saltos de línea múltiples por espacios
texto_limpio = re.sub(r"\s+", " ", texto)

# Quitamos espacios al inicio y al final
texto_limpio = texto_limpio.strip()

# Verificamos el resultado
print(texto_limpio[:1000])

The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer. Text file corrections and new HTML file by Joaquin Cuenca Abela. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra, tasaron cada pliego del dicho libro a t


## Segmentación en oraciones

El texto se divide en oraciones porque los modelos de lenguaje trabajan con secuencias.  
Cada oración será tratada como una secuencia independiente de palabras.

In [6]:
# Dividimos el texto en oraciones usando puntos, signos de pregunta y exclamación
oraciones = re.split(r'(?<=[.!?¿¡])\s+', texto_limpio)

# Eliminamos oraciones vacías
oraciones = [oracion.strip() for oracion in oraciones if oracion.strip()]

# Mostramos cuántas oraciones se obtuvieron
print("Cantidad de oraciones:", len(oraciones))

# Ejemplo de algunas oraciones
for i in range(5):
    print(f"{i+1}.", oraciones[i])

Cantidad de oraciones: 9577
1. ﻿The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.
2. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.net Title: Don Quijote Author: Miguel de Cervantes Saavedra Posting Date: April 27, 2010 [EBook #2000] Release Date: December, 1999 Language: Spanish *** START OF THIS PROJECT GUTENBERG EBOOK DON QUIJOTE *** Produced by an anonymous Project Gutenberg volunteer.
3. Text file corrections and new HTML file by Joaquin Cuenca Abela.
4. El ingenioso hidalgo don Quijote de la Mancha TASA Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de los que residen en su Consejo, certifico y doy fe que, habiendo visto por los señores dél un libro intitulado El ingenioso hidalgo de la Mancha, compuesto por Miguel de Cervantes Saavedra,

## Tokenización

Cada oración se convierte en una lista de tokens.  
En este caso, se usan minúsculas para reducir el tamaño del vocabulario, pero no se eliminan palabras ni se cambia su forma original mediante lematización.

In [7]:
# Función para tokenizar una oración en palabras
def tokenizar(oracion):
    # Convertimos a minúsculas para reducir variaciones del vocabulario
    oracion = oracion.lower()
    
    # Extraemos palabras y algunos signos de puntuación como tokens
    tokens = re.findall(r'\w+|[^\w\s]', oracion, re.UNICODE)
    
    return tokens

In [8]:
# Probamos la tokenización con una oración de ejemplo
ejemplo = oraciones[0]
tokens_ejemplo = tokenizar(ejemplo)

print("Oración original:")
print(ejemplo)

print("\nTokens:")
print(tokens_ejemplo)

Oración original:
﻿The Project Gutenberg EBook of Don Quijote, by Miguel de Cervantes Saavedra This eBook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.

Tokens:
['\ufeff', 'the', 'project', 'gutenberg', 'ebook', 'of', 'don', 'quijote', ',', 'by', 'miguel', 'de', 'cervantes', 'saavedra', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.']


## Tokens de inicio y fin

A cada oración se le agrega el token `<s>` al inicio y el token `</s>` al final.  
Esto permite que el modelo aprenda cómo suelen iniciar y terminar las oraciones.

In [9]:
# Tokenizamos todas las oraciones y agregamos tokens de inicio y fin
oraciones_tokenizadas = []

for oracion in oraciones:
    tokens = tokenizar(oracion)
    
    if len(tokens) > 0:
        tokens_con_marcas = ["<s>"] + tokens + ["</s>"]
        oraciones_tokenizadas.append(tokens_con_marcas)

# Mostramos un ejemplo
print(oraciones_tokenizadas[0])

['<s>', '\ufeff', 'the', 'project', 'gutenberg', 'ebook', 'of', 'don', 'quijote', ',', 'by', 'miguel', 'de', 'cervantes', 'saavedra', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '.', '</s>']


## División del corpus

El corpus se divide aleatoriamente en tres conjuntos: entrenamiento, validación y prueba.  
El conjunto de entrenamiento se usa para construir los modelos, validación para comparar resultados y prueba para evaluar el mejor modelo al final.

In [10]:
# Fijamos una semilla para que los resultados sean reproducibles
random.seed(42)

# Primero separamos entrenamiento y un conjunto temporal
train_sentences, temp_sentences = train_test_split(
    oraciones_tokenizadas,
    test_size=0.20,
    random_state=42
)

# Luego dividimos el temporal en validación y prueba
val_sentences, test_sentences = train_test_split(
    temp_sentences,
    test_size=0.50,
    random_state=42
)

print("Oraciones de entrenamiento:", len(train_sentences))
print("Oraciones de validación:", len(val_sentences))
print("Oraciones de prueba:", len(test_sentences))

Oraciones de entrenamiento: 7661
Oraciones de validación: 958
Oraciones de prueba: 958


## Vocabulario de entrenamiento

El vocabulario se construye usando únicamente el conjunto de entrenamiento.  
Esto permite medir después cuántas palabras del conjunto de prueba no fueron vistas durante el entrenamiento.

In [11]:
# Unimos todos los tokens del conjunto de entrenamiento
tokens_train = []

for oracion in train_sentences:
    tokens_train.extend(oracion)

# Creamos el vocabulario como conjunto de palabras únicas
vocab_train = set(tokens_train)

print("Tamaño del vocabulario de entrenamiento:", len(vocab_train))

Tamaño del vocabulario de entrenamiento: 21150


## Palabras no vistas en prueba

Se calcula cuántas palabras del conjunto de prueba no aparecen en el vocabulario del conjunto de entrenamiento.  
Estas palabras se conocen como OOV, que significa palabras fuera del vocabulario.

In [12]:
# Unimos todos los tokens del conjunto de prueba
tokens_test = []

for oracion in test_sentences:
    tokens_test.extend(oracion)

# Contamos cuántos tokens de prueba no están en el vocabulario de entrenamiento
oov_tokens = [token for token in tokens_test if token not in vocab_train]

# Calculamos la proporción de palabras no vistas
proporcion_oov = len(oov_tokens) / len(tokens_test)

print("Total de tokens en prueba:", len(tokens_test))
print("Tokens OOV:", len(oov_tokens))
print("Proporción OOV:", proporcion_oov)
print("Proporción OOV (%):", proporcion_oov * 100)

Total de tokens en prueba: 46397
Tokens OOV: 1363
Proporción OOV: 0.029376899368493654
Proporción OOV (%): 2.9376899368493654


### Relación entre palabras no vistas y data sparsity

Las palabras nunca vistas son aquellas que aparecen en el conjunto de prueba, pero no aparecieron en el conjunto de entrenamiento. Esto representa un problema porque el modelo no tiene información previa para calcular una probabilidad confiable para esas palabras.

Este problema se relaciona con la dispersión de datos, porque en lenguaje natural existen muchísimas palabras y combinaciones posibles, pero muchas aparecen muy pocas veces o no aparecen en el corpus de entrenamiento. Entonces, aunque el texto sea grande, no siempre contiene todos los casos posibles.

En los modelos de lenguaje basados en conteos, esto puede causar que algunas palabras o secuencias tengan conteo cero. Como consecuencia, el modelo puede asignar probabilidad cero a una oración, aunque esa oración sí sea válida en el idioma.